In [1]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

In [2]:
data = load_dataset('jxie/flickr8k')


README.md:   0%|          | 0.00/687 [00:00<?, ?B/s]

data/train-00000-of-00002-2f8f6bfa852eac(…): reconstructing file:   0%|          |  0.00B /  414MB            

data/train-00000-of-00002-2f8f6bfa852eac(…): downloading bytes:           |  0.00B            

data/train-00001-of-00002-2173151d8cd6c7(…): reconstructing file:   0%|          |  0.00B /  414MB            

data/train-00001-of-00002-2173151d8cd6c7(…): downloading bytes:           |  0.00B            

...

data/test-00000-of-00001-42a2661d12c73e4(…): reconstructing file:   0%|          |  0.00B /  137MB            

data/test-00000-of-00001-42a2661d12c73e4(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [3]:
print(data.shape)
print()
print(data)

{'train': (6000, 6), 'validation': (1000, 6), 'test': (1000, 6)}

DatasetDict({
    train: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
        num_rows: 6000
    })
    validation: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
        num_rows: 1000
    })
})


In [4]:
train = data['train']
test = data['test']
validation = data['validation']

print(train)
print(train.shape)
print(test.shape)
print(validation.shape)

Dataset({
    features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
    num_rows: 6000
})
(6000, 6)
(1000, 6)
(1000, 6)


In [5]:
train = pd.DataFrame({
    'img': train['image'],
    'caption_0': train['caption_0'],
    'caption_1': train['caption_1'],
    'caption_2': train['caption_2'],
    'caption_3': train['caption_3'],
    'caption_4': train['caption_4'],
})

test = pd.DataFrame({
    'img': test['image'],
    'caption_0': test['caption_0'],
    'caption_1': test['caption_1'],
    'caption_2': test['caption_2'],
    'caption_3': test['caption_3'],
    'caption_4': test['caption_4'],
})

validation = pd.DataFrame({
    'img': validation['image'],
    'caption_0': validation['caption_0'],
    'caption_1': validation['caption_1'],
    'caption_2': validation['caption_2'],
    'caption_3': validation['caption_3'],
    'caption_4': validation['caption_4'],
})


In [6]:
type(train)

pandas.core.frame.DataFrame

### **Text preprocessing**

In [7]:
!pip install cleantext

from cleantext import clean
from tensorflow.keras.layers import TextVectorization

In [8]:
def text_clean(txt):
    txt = clean(
        txt,
        lowercase = True,
        punct = True,
        extra_spaces = True
    )

    return f"<start> {txt} <end>"


def text_clean_2(df):
    for i in ['caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4']:
        df[i] = df[i].map(text_clean)                   # this sends the data sample by sample to the function

    return df


train = text_clean_2(train)                      # this sends the entire dataframe to the function
test = text_clean_2(test)
validation = text_clean_2(validation)

In [9]:
train['caption_0'].shape

(6000,)

In [10]:
train.head(3)

,img,caption_0,caption_1,caption_2,caption_3,caption_4
0,<PIL.JpegImagePlugin.JpegImageFile image mode=...,<start> a black dog is running after a white d...,<start> black dog chasing brown dog through sn...,<start> two dogs chase each other across the s...,<start> two dogs play together in the snow <end>,<start> two dogs running through a low lying b...
1,<PIL.JpegImagePlugin.JpegImageFile image mode=...,<start> a little baby plays croquet <end>,<start> a little girl plays croquet next to a ...,<start> the child is playing croquette by the ...,<start> the kid is in front of a car with a pu...,<start> the little boy is playing with a croqu...
2,<PIL.JpegImagePlugin.JpegImageFile image mode=...,<start> a brown dog in the snow has something ...,<start> a brown dog in the snow holding a pink...,<start> a brown dog is holding a pink shirt in...,<start> a dog is carrying something pink in it...,<start> a dog with something pink in its mouth...


In [11]:
def make_lists(df):
    imgs, captions = [], []
    
    for sample in range(len(df)):
        for i in range(5):
            captions.append(df[f"caption_{i}"].iloc[sample])
            imgs.append(df['img'].iloc[sample])

    return imgs, captions


train_imgs, train_captions = make_lists(train)
test_imgs, test_captions = make_lists(test)
validation_imgs, validation_captions = make_lists(validation)


print(len(train_imgs))
print(len(train_captions))
print()
print(len(test_imgs))
print(len(test_captions))
print()
print(len(validation_imgs))
print(len(validation_captions))

30000
30000

5000
5000

5000
5000


In [12]:
max_len = max(len(i.split()) for i in train_captions)
print(f"CAPTION HOLDING MAXIMUM LENGTH: {max_len}")

vectorizer = TextVectorization(
    max_tokens=None, output_mode='int', output_sequence_length=max_len
)

vectorizer.adapt(train_captions)                      # this makes a dictionary of vocab

vocab = vectorizer.get_vocabulary()
print(f"REAL VOCABULARY SIZE: {len(vocab)}")  # Will be ~7,632!

train_captions = vectorizer(train_captions).numpy()
test_captions = vectorizer(test_captions).numpy()
validation_captions = vectorizer(validation_captions).numpy()

print()
print("Train shape:", train_captions.shape)
print("Test shape:", test_captions.shape)
print("Validation shape:", validation_captions.shape)

CAPTION HOLDING MAXIMUM LENGTH: 38


I0000 00:00:1792086338.270389      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792086338.273260      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


REAL VOCABULARY SIZE: 7632

Train shape: (30000, 38)
Test shape: (5000, 38)
Validation shape: (5000, 38)


In [13]:
test_captions

array([[ 4,  6, 32, ...,  0,  0,  0],
       [ 4,  6, 32, ...,  0,  0,  0],
       [ 4, 14, 26, ...,  0,  0,  0],
       ...,
       [ 4,  2, 26, ...,  0,  0,  0],
       [ 4,  2, 10, ...,  0,  0,  0],
       [ 4,  2, 10, ...,  0,  0,  0]])

### **Pre-Trained model feature extraction**

In [14]:
import torch as tr
from transformers import ViTImageProcessor, ViTModel

In [15]:
if tr.cuda.is_available():
    device = tr.device('cuda')


processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')           # for extracting or using the features from model
model = ViTModel.from_pretrained('google/vit-base-patch16-224')                        # for using the model
model = model.to(device)
model.eval()


def img_vectorizer(images):
    img_vector = []
    
    for i in images:
        img = i.convert('RGB')
        inputs = processor(images=img, return_tensors='pt').to(device)

        with tr.no_grad():
            outputs = model(**inputs)

        vec = outputs.pooler_output.detach().cpu().numpy()[0]
        img_vector.append(vec)
        
    return img_vector


train_imgs = img_vectorizer(train['img'])
test_imgs = img_vectorizer(test['img'])
validation_imgs = img_vectorizer(validation['img'])

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
print(len(train_imgs))
print(len(test_imgs))
print(len(validation_imgs))

len(train_imgs[0])                           # each img is of the shape of 1 x 768 dimensions

6000
1000
1000


768

### **Transformer decoder architecture**

In [17]:
vocab = vectorizer.get_vocabulary()
len_1 = len(vocab)
len_2 = len(train_captions[0]) - 1

print(f"LENGTH OF IMAGE INPUTS : {len(train_imgs[0])}")
print(f"LENGTH OF CAPTION INPUTS : {len_2}")
print(f"LENGTH OF THE VOCABULARY : {len_1}")

LENGTH OF IMAGE INPUTS : 768
LENGTH OF CAPTION INPUTS : 37
LENGTH OF THE VOCABULARY : 7632


In [18]:
train_imgs[0].shape

(768,)

In [22]:
import keras_nlp
from tensorflow.keras.callbacks import EarlyStopping

In [19]:
# ====================================================================================================================
# making the caption embedding (embedding + positional embedding)

inputs = layers.Input(shape=(len_2,))
embedding = layers.Embedding(input_dim=len_1, output_dim=256)(inputs)
positional_embedding = keras_nlp.layers.PositionEmbedding(sequence_length=len_2)(embedding)
caption_embedding = embedding + positional_embedding

# ====================================================================================================================
# taking img and caption inputs

inputs_image = layers.Input(shape=(768,))
inputs_captions = layers.Input(shape=(len_2,))

# ====================================================================================================================
# converting imgs into the vector dimension of 256

img_vector_1 = layers.Dense(256, activation='relu')(inputs_image)                 # here we are producing 256 shape vector coz we have caption embedding of 256 dimensions, so to do future dot product of img and the caption embeddings we have to convert the img into the same dimension as the dimension of caption embeddings
img_vector_2 = layers.Reshape((1, 256))(img_vector_1) 

# ====================================================================================================================

x = keras_nlp.layers.TransformerDecoder(intermediate_dim=512, num_heads=6, dropout=0.2)(caption_embedding, encoder_sequence=img_vector_2)

# ====================================================================================================================

outputs = layers.Dense(len_1, activation='softmax')(x)





model = Model(
    inputs = [inputs_image, inputs_captions],
    outputs = outputs
)


model.compile(
    optimizer = tf.keras.optimizers.Adam(),
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics = ['accuracy']
)







NameError: name 'keras_nlp' is not defined